# Módulo de Rostro — Py-Feat en Colab (tarea 5 + 6)

Reescala las imágenes a **128×128 en RAM** (sin escribir el dataset en Drive) y corre **Py-Feat** sobre ellas, guardando un único CSV con: detección, confianza, pose (yaw/pitch/roll), 68 landmarks y 20 AUs.

**Antes de empezar:**
1. `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **GPU** (T4).
2. Ejecuta las celdas en orden.

El reescalado se hace en `/dev/shm` (un sistema de archivos montado en **RAM**, no en disco ni en Drive). Cada lote se reescala, se procesa y se libera. Lo único que queda guardado en Drive es el CSV final.

In [ ]:
# 1) Instalar Py-Feat
!pip install -q py-feat

In [ ]:
# 2) Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Configuración e imports
import os, time, shutil
from pathlib import Path

import torch
import pandas as pd
from PIL import Image

# Ruta del dataset en Drive (carpeta raíz; se busca recursivamente).
BASE_PATH = "/content/drive/MyDrive/Proyecto PIA 2026-I/Desarrollo/Datasets/rostro_dataset/AffectNet_DATASET_ESCOGIDO"

# CSV de salida (esto SÍ se guarda en Drive: es el resultado de la tarea 6).
OUTPUT_CSV = "/content/drive/MyDrive/Proyecto PIA 2026-I/Desarrollo/Datasets/rostro_dataset/pyfeat_features_v5.csv"

TARGET_SIZE = (128, 128)   # resolución objetivo (ancho, alto)
RAM_DIR = "/dev/shm/affnet_128"  # carpeta temporal EN RAM (tmpfs)
CHUNK = 500                # imágenes por lote (reescalar -> Py-Feat -> liberar)

# GPU automática: usa CUDA si Colab asignó GPU, si no CPU.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH = 32 if DEVICE == "cuda" else 8
print("Dispositivo:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("AVISO: sin GPU. Activa: Entorno de ejecución -> Cambiar tipo -> GPU")

In [ ]:
# 4) Columnas que se guardan en el CSV
AU_COLUMNS = [
    "AU01", "AU02", "AU04", "AU05", "AU06", "AU07", "AU09", "AU10", "AU11",
    "AU12", "AU14", "AU15", "AU17", "AU20", "AU23", "AU24", "AU25", "AU26",
    "AU28", "AU43",
]
LANDMARK_COLUMNS = [f"x_{i}" for i in range(68)] + [f"y_{i}" for i in range(68)]
DETECCION_COLUMNS = ["FaceRectX", "FaceRectY", "FaceRectWidth", "FaceRectHeight", "FaceScore"]
POSE_COLUMNS = ["Pitch", "Roll", "Yaw"]
FEATURE_COLUMNS = DETECCION_COLUMNS + POSE_COLUMNS + LANDMARK_COLUMNS + AU_COLUMNS

In [ ]:
# 5) Descubrir todas las imágenes del dataset (recursivo)
EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
imagenes = sorted(p for p in Path(BASE_PATH).rglob("*") if p.suffix.lower() in EXTS)
print("Imágenes encontradas:", len(imagenes))
assert imagenes, "No se encontraron imágenes. Revisa BASE_PATH."

In [ ]:
# 6) Construir el detector (sin identidad ni gaze: el proyecto no los usa)
os.environ["TQDM_DISABLE"] = "1"  # silencia las barras internas de Py-Feat
from feat import Detectorv1

detector = Detectorv1(device=DEVICE, identity_model=None, gaze_model=None)
print("Detector listo en", DEVICE)

In [ ]:
# 7) Helper: reducir el Fex de Py-Feat a las columnas pedidas
def fex_a_df(fex, mapping):
    cols = [c for c in FEATURE_COLUMNS if c in fex.columns]
    df = fex.loc[:, cols].copy()
    # 'input' trae la ruta en /dev/shm; la mapeamos a la ruta relativa original.
    df.insert(0, "imagen", [mapping[Path(v).name] for v in fex["input"]])
    df.insert(1, "face_detected", df["FaceScore"].notna())
    return df

In [ ]:
# 8) Reescalar EN RAM por lotes y correr Py-Feat (con progreso, velocidad y ETA)
resultados = []
errores = []
total = len(imagenes)
hechas = 0
t0 = time.time()

for inicio in range(0, total, CHUNK):
    lote = imagenes[inicio:inicio + CHUNK]
    # Carpeta RAM limpia para este lote.
    shutil.rmtree(RAM_DIR, ignore_errors=True)
    os.makedirs(RAM_DIR, exist_ok=True)

    mapping = {}
    shm_paths = []
    for j, ruta in enumerate(lote):
        nombre = f"{inicio + j:07d}.jpg"
        try:
            with Image.open(ruta) as im:
                im = im.convert("RGB").resize(TARGET_SIZE, Image.Resampling.LANCZOS)
                destino = os.path.join(RAM_DIR, nombre)
                im.save(destino, format="JPEG", quality=95)
            mapping[nombre] = str(ruta.relative_to(BASE_PATH))
            shm_paths.append(destino)
        except Exception as e:
            errores.append((str(ruta), repr(e)))

    if shm_paths:
        fex = detector.detect(shm_paths, data_type="image", batch_size=BATCH, progress_bar=False)
        resultados.append(fex_a_df(fex, mapping))

    hechas += len(lote)
    elapsed = time.time() - t0
    vel = hechas / elapsed if elapsed else 0
    eta = (total - hechas) / vel if vel else 0
    print(f"{hechas}/{total} ({100 * hechas / total:5.1f}%) | {vel:5.1f} img/s | "
          f"transcurrido {elapsed / 60:5.1f} min | ETA {eta / 60:5.1f} min", flush=True)

shutil.rmtree(RAM_DIR, ignore_errors=True)
df = pd.concat(resultados, ignore_index=True) if resultados else pd.DataFrame()
print("\nListo. Filas:", len(df), "| Errores de lectura:", len(errores))

In [ ]:
# 9) Guardar el CSV en Drive y resumen
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)

faltantes = [c for c in FEATURE_COLUMNS if c not in df.columns]
print("CSV guardado en:", OUTPUT_CSV)
print("Columnas:", len(df.columns), "| Filas:", len(df))
print("Con rostro detectado:", int(df["face_detected"].sum()), "/", len(df))
if faltantes:
    print("OJO, columnas esperadas ausentes (revisar nombres de Py-Feat):", faltantes)
df.head()